# Reliability and Calibration of ML-based NIDS
## Under Cross-Dataset Distribution Shift
**CSE499B | Section 15 | Group 02**

| Name | ID |
|---|---|
| Kazi Safin Arafat | 2211778642 |
| Khondokar Sajid | 2211954042 |
| Moushumi Akter Mow | 2021983642 |
| Rakibul Hasan Ridoy | 1731339042 |

---
**Update Report 03 — Model Training, Performance Evaluation, and Calibration Analysis**

---
## Section 1: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, brier_score_loss,
    roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print('All libraries loaded successfully.')

---
## Section 2: Load and Inspect Dataset
We load the combined CIC-IDS-2017 CSV file and perform an initial inspection of its shape and label distribution.

In [ ]:
df = pd.read_csv('combine.csv')
df.columns = df.columns.str.strip()

print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'\nLabel distribution (raw):')
print(df['Label'].value_counts().to_string())

---
## Section 3: Preprocessing
Steps performed:
- Remove duplicate records
- Replace infinite values with NaN and drop affected rows
- Convert multi-class labels to binary (Benign = 0, Attack = 1)
- Normalize numerical features using StandardScaler

In [ ]:
# Drop duplicates
df = df.drop_duplicates()

X_raw = df.drop('Label', axis=1)
y_raw = df['Label']

# Remove infinite and NaN values
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
mask = X_raw.notna().all(axis=1)
X_raw = X_raw[mask]
y_raw = y_raw[mask]

print(f'Rows retained after cleaning : {X_raw.shape[0]:,}')
print(f'Rows removed                 : {df.shape[0] - X_raw.shape[0]:,}')

# Binary label encoding
y = y_raw.apply(lambda x: 0 if str(x).upper() == 'BENIGN' else 1)
print(f'\nBinary label distribution:')
print(y.value_counts().rename({0: 'Benign (0)', 1: 'Attack (1)'}).to_string())

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print(f'\nFeature matrix shape after scaling: {X_scaled.shape}')

---
## Section 4: Exploratory Data Analysis (EDA)
Visual inspection of class distribution, feature variance, and feature correlations.

In [ ]:
# 4a. Class distribution
label_counts = y.value_counts().rename({0: 'Benign', 1: 'Attack'})

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Dataset Overview', fontsize=13, fontweight='bold')

axes[0].bar(label_counts.index, label_counts.values,
            color=['steelblue', 'tomato'], width=0.5, edgecolor='white')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Sample Count')
axes[0].set_xlabel('Class')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 300, f'{v:,}', ha='center', fontsize=10)

# 4b. Top 20 features by standard deviation
feature_std = pd.Series(X_raw.std().values, index=X_raw.columns).nlargest(20)
axes[1].barh(feature_std.index[::-1], feature_std.values[::-1], color='steelblue')
axes[1].set_title('Top 20 Features by Standard Deviation')
axes[1].set_xlabel('Standard Deviation')

plt.tight_layout()
plt.show()

In [ ]:
# 4c. Correlation heatmap — top 15 features by variance
top_features = pd.Series(X_raw.std().values, index=X_raw.columns).nlargest(15).index
corr_matrix = X_raw[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 9))
fig.suptitle('Correlation Heatmap — Top 15 Features by Variance', fontsize=13, fontweight='bold')

sns.heatmap(
    corr_matrix, ax=ax, cmap='coolwarm', center=0,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.4, square=True
)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 4d. Distribution of top 6 features by class
top6 = pd.Series(X_raw.std().values, index=X_raw.columns).nlargest(6).index
X_eda = X_raw[top6].copy()
X_eda['Label'] = y.values

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Feature Distributions by Class (Top 6 Features)', fontsize=13, fontweight='bold')

for ax, feat in zip(axes.flatten(), top6):
    benign_vals = X_eda.loc[X_eda['Label'] == 0, feat]
    attack_vals = X_eda.loc[X_eda['Label'] == 1, feat]
    ax.hist(benign_vals, bins=40, alpha=0.6, color='steelblue', label='Benign', density=True)
    ax.hist(attack_vals, bins=40, alpha=0.6, color='tomato', label='Attack', density=True)
    ax.set_title(feat[:30], fontsize=9)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Section 5: Dataset Splitting
The dataset is split into training (80%) and test (20%) sets with stratification to preserve class ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set : {X_train.shape[0]:,} samples')
print(f'Test set     : {X_test.shape[0]:,} samples')

---
## Section 6: Model Training — Random Forest Classifier
A Random Forest with 100 estimators serves as the baseline classifier. It provides probabilistic outputs needed for calibration evaluation.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print('Model training complete.')

---
## Section 7: Performance Evaluation
Standard discriminative metrics: Accuracy, Precision, Recall, F1-score, and ROC-AUC.

In [ ]:
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('Model Performance on Test Set')
print('-' * 35)
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-score  : {f1:.4f}')
print(f'ROC-AUC   : {auc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack']))

In [ ]:
# Confusion matrix + ROC curve side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model Performance Visualizations', fontsize=13, fontweight='bold')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign', 'Attack'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'Random Forest (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Predicted probability distribution
fig, ax = plt.subplots(figsize=(9, 4))
fig.suptitle('Predicted Probability Distribution by True Class', fontsize=13, fontweight='bold')

ax.hist(y_prob[y_test == 0], bins=40, alpha=0.6, color='steelblue', label='True Benign', density=True)
ax.hist(y_prob[y_test == 1], bins=40, alpha=0.6, color='tomato', label='True Attack', density=True)
ax.set_xlabel('Predicted Probability (Attack)')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.show()

---
## Section 8: Calibration Evaluation
We evaluate how well the model's predicted probabilities reflect true outcome likelihoods using:
- **Expected Calibration Error (ECE)** — weighted average gap between confidence and accuracy across bins
- **Brier Score** — mean squared error between predicted probabilities and true labels
- **Reliability Diagram** — visual inspection of calibration

In [ ]:
def compute_ece(y_true, y_prob, n_bins=10):
    """Expected Calibration Error (ECE)."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)
    ece = 0.0
    for i in range(n_bins):
        mask = bin_ids == i
        if np.sum(mask) > 0:
            acc  = np.mean(y_true[mask])
            conf = np.mean(y_prob[mask])
            ece += np.abs(acc - conf) * np.sum(mask) / len(y_true)
    return ece

brier = brier_score_loss(y_test, y_prob)
ece   = compute_ece(y_test.values, y_prob)

print('Calibration Metrics (In-Domain)')
print('-' * 35)
print(f'Expected Calibration Error (ECE) : {ece:.4f}')
print(f'Brier Score                      : {brier:.4f}')
print()
print('Interpretation:')
print('  ECE close to 0 indicates well-calibrated confidence estimates.')
print('  Brier Score close to 0 indicates accurate probability predictions.')

In [ ]:
# Reliability diagram + per-bin calibration gap
prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)

bins_arr = np.linspace(0, 1, 11)
bin_ids  = np.digitize(y_prob, bins_arr) - 1
bin_ids  = np.clip(bin_ids, 0, 9)
bin_counts = np.array([np.sum(bin_ids == i) for i in range(10)])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Calibration Analysis', fontsize=13, fontweight='bold')

# Reliability diagram
axes[0].plot(prob_pred, prob_true, marker='o', color='steelblue', lw=2, label='Random Forest')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect Calibration')
axes[0].fill_between(prob_pred, prob_pred, prob_true, alpha=0.15, color='tomato', label='Calibration Gap')
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Fraction of Positives')
axes[0].set_title('Reliability Diagram')
axes[0].legend(fontsize=9)

# Per-bin calibration gap bar chart
gaps = np.abs(prob_true - prob_pred)
bin_centers = 0.5 * (bins_arr[:-1] + bins_arr[1:])
gap_bins = bin_centers[:len(gaps)]
axes[1].bar(gap_bins, gaps, width=0.08, color='tomato', align='center', edgecolor='white')
axes[1].set_xlabel('Confidence Bin')
axes[1].set_ylabel('|Accuracy - Confidence|')
axes[1].set_title('Per-Bin Calibration Gap')
axes[1].axhline(y=ece, color='steelblue', linestyle='--', label=f'ECE = {ece:.4f}')
axes[1].legend(fontsize=9)

# Bin sample counts
axes[2].bar(bins_arr[:10] + 0.05, bin_counts, width=0.08, color='steelblue', edgecolor='white')
axes[2].set_xlabel('Confidence Bin')
axes[2].set_ylabel('Sample Count')
axes[2].set_title('Sample Count per Confidence Bin')

plt.tight_layout()
plt.show()

In [ ]:
# Summary metrics bar chart
metrics = {
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1-score': f1,
    'ROC-AUC': auc,
    'Brier Score': brier,
    'ECE': ece
}
colors = ['steelblue'] * 5 + ['tomato', 'tomato']

fig, ax = plt.subplots(figsize=(10, 4))
fig.suptitle('Summary — Discriminative and Calibration Metrics (In-Domain)', fontsize=13, fontweight='bold')

bars = ax.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='white')
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_xlabel('Metric')
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=0.8)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Discriminative Metrics'),
    Patch(facecolor='tomato', label='Calibration Metrics (lower is better)')
]
ax.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.show()

---
## Section 9: Feature Importance
Top features contributing to the Random Forest classifier's decisions.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_raw.columns).nlargest(20)

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Top 20 Feature Importances — Random Forest', fontsize=13, fontweight='bold')

ax.barh(importances.index[::-1], importances.values[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_ylabel('Feature')

plt.tight_layout()
plt.show()

---
## Section 10: Summary and Next Steps

**Results so far (In-Domain — CIC-IDS-2017):**

| Metric | Value |
|---|---|
| Accuracy | computed above |
| F1-score | computed above |
| ROC-AUC | computed above |
| ECE | computed above |
| Brier Score | computed above |

**Observations:**
- The Random Forest achieves strong discriminative performance on the in-domain test set.
- Calibration metrics (ECE, Brier Score) provide the baseline reliability measurements for the in-domain setting.
- The reliability diagram reveals whether the model is over- or under-confident in its predictions.

**Next steps (Update 04):**
1. Cross-dataset evaluation: train on CIC-IDS-2017, deploy on UNSW-NB15 or NSL-KDD
2. Measure calibration degradation (ECE / Brier Score increase) under distribution shift
3. Apply post-hoc recalibration: Temperature Scaling and Isotonic Regression
4. Implement entropy-based OOD rejection gate (UNKNOWN class)
5. Evaluate selective accuracy and UNKNOWN rate trade-offs